[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_mnist_cosine_similarity_didattica.ipynb)

# MNIST — Riconoscere cifre scritte a mano usando la similarità coseno

## Cosa abbiamo già visto a lezione

1. **Vettori** (presentazione *ripasso_vettori*) — somma, prodotto scalare, norma, **similarità coseno**.
2. **Parametri** (presentazione *parametri*) — un modello di AI è un insieme di numeri imparati dai dati.
3. **Training** e **inferenza** — prima si imparano i parametri, poi si usano per fare predizioni.

## Cosa faremo oggi

Costruiremo da zero un classificatore di cifre scritte a mano. Useremo solo gli strumenti della lezione sui vettori — niente reti neurali, niente magia. L'idea è elementare:

- per ogni cifra (0…9) calcoliamo la sua **immagine media** sul training set;
- per riconoscere una cifra nuova, vediamo a quale delle 10 immagini medie **somiglia di più** (similarità coseno).

Il notebook è pensato per chi non conosce Python: tutta la sintassi tecnica è già nascosta in funzioni con nomi italiani.

## Setup (eseguire una volta)

In [ ]:
!pip install --upgrade --no-cache-dir oli_ai > /dev/null
from oli_ai.mnist_lib import *

## 1. Richiamo delle formule

Nelle slide *ripasso vettori* abbiamo visto che la similarità coseno fra due vettori $\vec{a}$ e $\vec{b}$ si calcola in tre passi.

**Prodotto scalare** (dot product):

$$\vec{a}\cdot\vec{b} \;=\; \sum_{i} a_i \, b_i$$

**Norma** (lunghezza del vettore):

$$\|\vec{a}\| \;=\; \sqrt{\sum_{i} a_i^{\,2}}$$

**Similarità coseno**:

$$\text{sim}(\vec{a},\vec{b}) \;=\; \frac{\vec{a}\cdot\vec{b}}{\|\vec{a}\|\;\|\vec{b}\|}$$

Vale **1** se i vettori hanno la stessa direzione, **0** se sono perpendicolari, **−1** se sono opposti.

Traduciamo ora le tre formule in codice — letteralmente, riga per riga.

> **Sintassi.** In questo notebook scriviamo un vettore con la notazione `v[1, 2, 3]`, molto vicina alla notazione matematica $\vec{v} = (1, 2, 3)$.

In [ ]:
import math

def dot(a, b):
    somma = 0
    for i in range(len(a)):
        somma = somma + a[i] * b[i]
    return somma

def norm(a):
    somma = 0
    for i in range(len(a)):
        somma = somma + a[i] * a[i]
    return math.sqrt(somma)

def sim(a, b):
    return dot(a, b) / (norm(a) * norm(b))


# --- Verifichiamo le formule su qualche vettore di esempio ---

a = v[1, 2, 3]
b = v[2, 4, 6]      # b = 2 * a, stessa direzione
print('a · b      =', dot(a, b))
print('||a||      =', norm(a))
print('||b||      =', norm(b))
print('sim(a, b)  =', sim(a, b))      # = 1   (paralleli)

c = v[1, 0]
d = v[0, 1]
print('sim(c, d)  =', sim(c, d))      # = 0   (perpendicolari)

e = v[1, 0]
f = v[-1, 0]
print('sim(e, f)  =', sim(e, f))      # = -1  (opposti)

Le funzioni `dot`, `norm` e `sim` che abbiamo appena scritto **sono** la similarità coseno: niente di più, niente di meno.

Da qui in poi useremo le versioni equivalenti già pronte nella libreria — `prodotto_scalare`, `norma`, `similarità` — perché su 10.000 immagini sono migliaia di volte più veloci. Ma quello che fanno è *esattamente* il codice qui sopra.

## 2. Il dataset MNIST: training set e test set

MNIST contiene **70.000 cifre scritte a mano**, già divise in due gruppi separati:

- **training set** (60.000 immagini) — i dati che useremo per **addestrare** il modello (cioè per calcolare i 10 vettori medi).
- **test set** (10.000 immagini) — dati che il modello **non vedrà** durante il training. Li teniamo da parte e li useremo solo alla fine, per valutare il modello.

### Ogni immagine è etichettata

A ogni immagine del dataset è associata la cifra che rappresenta (un numero da 0 a 9): questa è la sua **etichetta**. Le etichette sono state assegnate **a mano**, una volta sola, da chi ha costruito il dataset. Ci servono per due cose:

- **Durante il training**: usiamo le etichette per raggruppare le immagini per cifra — per esempio, prendere *tutte le immagini etichettate `5`* e calcolarne la media. Senza etichette non sapremmo quali immagini mettere insieme.
- **Durante la valutazione**: usiamo le etichette del test set come "risposte giuste" per misurare se il modello ha indovinato.

In Python ogni gruppo arriva quindi diviso in due liste parallele: le **immagini** (`immagini_train`, `immagini_test`) e le **etichette** corrispondenti (`etichette_train`, `etichette_test`). La n-esima immagine ha la n-esima etichetta.

### Perché separare training e test?

Quando valutiamo un modello **non dobbiamo barare**: dobbiamo testarlo su dati che non ha mai visto, per stimare onestamente come si comporterebbe nel **mondo reale** di fronte a cifre nuove. Un modello che funziona solo sulle stesse immagini con cui è stato addestrato non serve a niente: vogliamo che sia capace di **generalizzare** — di applicare ciò che ha imparato dai dati di training a casi nuovi. L'accuratezza misurata sul test set è la nostra unica stima onesta di questa capacità.

### Da matrice a vettore

Ogni immagine è una **matrice 28×28** di numeri da 0 (bianco) a 255 (nero). Ma la similarità coseno la sappiamo calcolare solo fra **vettori**, non fra matrici. Come facciamo?

La risposta è semplice: trasformiamo la matrice in vettore **appiattendola**, cioè concatenando le 28 righe una dopo l'altra. Otteniamo un vettore lungo 28 × 28 = **784 numeri**. Per esempio, ecco come si appiattisce una piccola matrice 3×3:

```
[1 2 3]
[4 5 6]   →   [1, 2, 3, 4, 5, 6, 7, 8, 9]
[7 8 9]
```

L'operazione è **completamente reversibile**: dato un vettore di 784 numeri possiamo ricostruire la matrice 28×28 ritagliandolo in 28 segmenti da 28 numeri ciascuno (il primo segmento diventa la riga 0, il secondo la riga 1, e così via). Matrice e vettore appiattito sono **lo stesso oggetto**, scritto in due modi diversi.

Nel notebook **non dobbiamo appiattire a mano**: le funzioni della libreria (`similarità`, `predici`, …) accettano sia matrici 28×28 sia vettori da 784 numeri, e si occupano dell'appiattimento internamente. Ma è importante sapere che dietro c'è solo questa trasformazione — la similarità coseno fra due immagini è la similarità coseno fra le loro versioni appiattite.

In [ ]:
immagini_train, etichette_train, immagini_test, etichette_test = carica_mnist()

print('immagini di training:', len(immagini_train))
print('immagini di test:    ', len(immagini_test))
print('formato di una immagine:', immagini_train[0].shape, '(matrice 28x28)')

### Una cifra è una matrice di numeri

Visualizziamo la prima immagine del training set e leggiamo il valore di qualche pixel.

In [ ]:
prima_immagine = immagini_train[0]
mostra_cifra(prima_immagine)

print('etichetta vera:', etichette_train[0])
print('pixel in alto a sinistra (riga 0, colonna 0):', prima_immagine[0][0])
print('pixel quasi al centro  (riga 14, colonna 14):', prima_immagine[14][14])

Ecco una piccola galleria — varietà di calligrafie per la stessa cifra.

In [ ]:
mostra_cifre(immagini_train, etichette_train, righe=3, colonne=10)

## 3. Fase 1 — Training: calcoliamo i 10 vettori medi

L'idea: per ogni cifra (0…9) prendiamo **tutte** le sue copie nel training set e ne facciamo la **media pixel per pixel**. Otteniamo una specie di "prototipo" di quella cifra.

Vediamo prima il risultato per la sola cifra **5**.

In [ ]:
cifre_5 = filtra_per_cifra(immagini_train, etichette_train, cifra=5)
print('Quante cifre 5 ci sono nel training set?', len(cifre_5))

media_5 = media_immagini(cifre_5)
print('La cifra-prototipo della classe 5:')
mostra_cifra(media_5)

Bello, no? È un "5 medio", un po' sfumato perché è la media di tante calligrafie diverse.

Ora ripetiamo l'operazione per **tutte** le cifre da 0 a 9 e mettiamo i risultati in una lista che chiameremo `modello`.

In [ ]:
modello = []
for cifra in range(10):
    immagini_di_questa_cifra = filtra_per_cifra(immagini_train, etichette_train, cifra)
    vettore_medio = media_immagini(immagini_di_questa_cifra)
    modello.append(vettore_medio)

print('Il modello contiene', len(modello), 'vettori medi (uno per cifra):')
mostra_cifre(modello, list(range(10)), righe=2, colonne=5)

## 4. Quanti parametri ha il nostro modello?

Richiamando la presentazione sui *parametri*: i parametri sono **i numeri che il modello ha imparato dai dati**. Nel nostro caso:

- 10 vettori medi (uno per cifra)
- ogni vettore medio è una matrice 28×28 = **784 numeri**

Totale: **10 × 784 = 7.840 parametri**.

Per confronto:
- una rete neurale piccola per MNIST: ~100.000 parametri
- GPT-3: 175 **miliardi** di parametri

Il nostro modello è ridicolmente piccolo. Vediamo quanto va lontano comunque.

In [ ]:
numero_parametri = 10 * 28 * 28
print('Parametri del nostro modello:', numero_parametri)

## 5. Fase 2 — Inferenza: classificare una cifra nuova

Inferenza = **usare** il modello già addestrato su immagini mai viste.

**Algoritmo di predizione**: data un'immagine nuova, calcolo la similarità coseno fra l'immagine e ciascuno dei 10 vettori medi del modello. La cifra che ha il vettore medio **più simile** è la mia predizione.

Proviamolo su una sola immagine del test set.

In [ ]:
immagine_da_riconoscere = immagini_test[50]
mostra_cifra(immagine_da_riconoscere)
print('etichetta vera:    ', etichette_test[50])

# vediamo le 10 similarità una per una
for cifra in range(10):
    s = similarità(immagine_da_riconoscere, modello[cifra])
    print(f'  similarità con il prototipo della cifra {cifra}:  {s:.3f}')

print('predizione del modello:', predici(immagine_da_riconoscere, modello))

Notate che il valore di similarità più alto corrisponde alla cifra che il modello "sceglie" — è proprio così che funziona `predici`.

Ora applichiamolo a **tutte** le 10.000 immagini di test.

In [ ]:
predizioni = predici_tutte(immagini_test, modello)

# guardiamo le prime 15 predizioni
mostra_cifre(immagini_test, predizioni, righe=3, colonne=5)

## 6. Valutazione: quanto è bravo il modello?

L'**accuratezza** è la percentuale di cifre classificate correttamente sul test set.

In [ ]:
acc = accuratezza(predizioni, etichette_test)
print(f'Accuratezza sul test set: {acc*100:.1f}%')

**~82%** con un modello fatto di sole medie e similarità coseno: niente male per qualcosa che si scrive in 5 righe.

(Una rete neurale moderna raggiunge >99% — ma con migliaia o milioni di parametri.)

## 7. Dove sbaglia il modello?

L'accuratezza è un solo numero: ci dice *quanto* il modello sbaglia, non *come*. La **matrice di confusione** mostra, per ogni cifra reale, come si distribuiscono le predizioni del modello:

- sulle **righe** la cifra reale
- sulle **colonne** la cifra predetta
- sulla **diagonale** le predizioni corrette.

In [ ]:
mostra_matrice_confusione(etichette_test, predizioni)

Quali cifre sono più difficili da riconoscere?

In [ ]:
cifre_piu_difficili(etichette_test, predizioni)

E come sono fatti gli errori più frequenti? Vediamo qualche esempio.

In [ ]:
mostra_errori_frequenti(immagini_test, etichette_test, predizioni)

## Riepilogo

| Concetto della lezione | Realizzazione nel notebook |
|---|---|
| vettore | una cifra è un vettore di 784 numeri |
| similarità coseno | criterio di confronto fra cifre |
| parametri | i 10 vettori medi (7.840 numeri) |
| training | calcolo dei 10 vettori medi sul training set |
| inferenza | per ogni immagine nuova, scelgo il vettore medio più simile |

## Esercizi

1. Visualizzate qualche immagine **classificata male**: secondo voi, perché il modello si è confuso?
2. Quale coppia di cifre è la più "confondibile"? Guardate la matrice di confusione.
3. Provate a togliere dal training set tutte le immagini di una cifra (per esempio la 7). Cosa succede in fase di inferenza quando arriva un 7? Riuscite a prevederlo prima di provare?
4. (più difficile) La similarità coseno *non* dipende dalla "intensità" complessiva dell'immagine. Cosa succederebbe se due 8 fossero scritti uno con tratto sottile e uno con tratto spesso? E se cambiassimo metrica usando la **distanza euclidea** invece della similarità coseno?